In [ ]:
#take folder of folders (imageval), loop through to put all images w, h > wbound, hbound in selected merged directory
#for each image in directory, center crop to 1024 x 1024
#patchify to 4x4, collect patches, store in (big patch) (can repeat for 2x2, 1x1, 8x8)
#downsample to 512 and patchify to 4x4 again, store in (medium patch)
#downsample to 256 and patchify to 4x4 again, store in (small patch)
#normalize patch pixel values to (0, 1), by N(0, 1) or by max - min = 1
#calculate FD score

In [ ]:
import pathlib, shutil, os, math, csv, random
from PIL import Image
import numpy as np
import torch
from torchvision import transforms
from scipy.linalg import linalg

In [ ]:
#FUNCTIONS:
def select_images(wbound, hbound, image_root, out_dir):
  """
    wbound is minimum width allowed
    hbound is minimum height allowed
    image root is directory that images are stored in
    dest dir is directory to move stored images to

    Sends all images with dim > (wbound, hbound) to dest_dir
  """
  out_dir = pathlib.Path(out_dir)
  out_dir.mkdir(exist_ok = True)

  selected_counter = 0
  processed_counter = 0
  for p in pathlib.Path(image_root).rglob('*'):
      if p.is_file(): # passes directory
          try:
              w, h = Image.open(p).size
          except OSError:
              continue
          if (w >= wbound and h >= hbound):
              shutil.copy(p, out_dir/p.name)
          #TODO: REMOVE PRINT CHECKERS BELOW HERE -------
              selected_counter += 1
              print(f"{selected_counter} images selected")
              print("w: ", w, "h: ", h)
          processed_counter += 1
          if processed_counter % 10 == 0:
              print(f"{processed_counter} images processed")




def center_crop(src_dir, out_dir, size):
    """
    src_dir: folder of images of raw size
    out_dir: wanted folder of images (size x size)

    Crops all images in src_dir to (output_size x output_size) in out_dir
    """
    out_dir = pathlib.Path(out_dir)
    if out_dir.exists():
        print("out dir removed")
        shutil.rmtree(out_dir)
    out_dir.mkdir(exist_ok = True)
    crop = transforms.CenterCrop(size)
    counter = 0

    for p in pathlib.Path(src_dir).rglob('*'):
        if p.is_file():
            img = Image.open(p).convert("RGB")
            img_c = crop(img)
            img_c.save(out_dir/p.name)
            #TODO: REMOVE PRINT CHECKERS BELOW HERE --------
            counter += 1
            if counter % 10 == 0:
              print(f"{counter} images cropped")


#TODO: add different downsampling methods
def downsample(src_dir, out_dir, size):

    out_dir = pathlib.Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(exist_ok = True)
    transform = transforms.Resize(size, interpolation=Image.BICUBIC)
    counter = 0
    for p in pathlib.Path(src_dir).rglob('*'):
        if p.is_file():
            img = Image.open(p).convert("RGB")
            img_d = transform(img)
            img_d.save(out_dir/p.name)
            #TODO: REMOVE PRINT CHECKERS BELOW HERE --------
            counter += 1
            if counter % 10 == 0:
              print(f"{counter} images downsampled")


def patchify_normalize(src_dir, out_dir, ps, norm = "standard"):
  """
  patchify and normalize all the images in src_dir to collection of ps x ps patches in out_dir.

  """
  out_dir = pathlib.Path(out_dir)
  if out_dir.exists():
      shutil.rmtree(out_dir)
  out_dir.mkdir(exist_ok = True)
  counter = 0
  all_patches = []

  for p in pathlib.Path(src_dir).rglob('*'):
    if p.is_file(): # passes directory
        img = np.array(Image.open(p))
        H, W, C= img.shape
        assert H % ps == 0 and W % ps == 0 #image dimensions must be divisible by patch size
        patches = img.reshape(H//ps, ps, W//ps, ps, C)
        patches = patches.swapaxes(1, 2).reshape(-1, ps, ps, C) #many patches with C channels
        patches = patches.reshape(patches.shape[0], -1) # (npatches, ps*ps*c)
        patches = patches.astype(np.float32)
        if norm == "standard":
            stds = patches.std(axis = 1, keepdims = True)
            print("stds", stds)
            patches = (patches - patches.mean(axis = 1, keepdims = True)) / stds
        elif norm == "minmax":
            patches = (patches - np.min(patches)) / (np.max(patches) - np.min(patches))
        else:
            patches = patches/255.0

        all_patches.append(patches)
        #TODO: REMOVE PRINT CHECKERS BELOW HERE --------
        counter += 1
        if counter % 1 == 0:
            print(f"{counter} images patchified")
  return np.vstack(all_patches)


def frechet_distance(X, Y):
    """
    Takes two np arrays of shape (n, dim) X, Y
    Returns frechet distance between two distributions
    """
    mu1, mu2 = X.mean(axis = 0), Y.mean(axis = 0) #(48, )
    sigma1, sigma2 = X.std(axis = 0), Y.std(axis = 0) #(48, )

    cov_x = np.cov(X, rowvar = False)
    cov_y = np.cov(Y, rowvar = False)

    covmean = linalg.sqrtm(cov_x @ cov_y)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fd2 = np.sum((mu1 - mu2) **2) + np.trace(cov_x + cov_y - 2 * covmean)
    fd = np.sqrt(max(np.real(fd2), 0))/np.sqrt(X.shape[1])

    return fd



In [ ]:
#take folder of folders (imageval), loop through to put all images w, h > wbound, hbound in selected merged directory
#for each image in directory, center crop to 1024 x 1024
#patchify to 4x4, collect patches, store in (big patch) (can repeat for 2x2, 1x1, 8x8)
#downsample to 512 and patchify to 4x4 again, store in (medium patch)
#downsample to 256 and patchify to 4x4 again, store in (small patch)
#normalize patch pixel values to (0, 1), by N(0, 1) or by max - min = 1
#calculate FD score

In [ ]:
if __name__ == "__main__":

    #TODO: change to actual path
    image_dir = "/content/drive/MyDrive/UROP He Vision Group/imagenet-val" #FOLDER OF FOLDERS OF IMAGES
    out_dir = "/content/drive/MyDrive/UROP He Vision Group/imagenet_temp_folders"

    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(exist_ok = True)

    resolutions = [1024, 512, 256, 128]
    patch_vec = {}

    #TODO: if you want, iterate over possible patch sizes
    #not used: patch_sizes = [16, 8, 4, 2]

    select_images(1024, 1024, image_dir, f"{out_dir}/large_images")
    print("images selected")
    center_crop(f"{out_dir}/large_images", f"{out_dir}/resized_images", 1024)
    print("images cropped")

    for res in resolutions:
        downsample(f"{out_dir}/resized_images", f"{out_dir}/downsampled_{res}", res)
        print(f"images downsampled to {res}")

        patch_vec[res] = patchify_normalize(f"{out_dir}/downsampled_{res}", f"{out_dir}/patches_{res}", 4)
        print(f"images patchified + normalized at {res}")

    print(frechet_distance(patch_vec[1024], patch_vec[512]))